# 🎙️ Transcritor de áudio em português (Whisper large-v3 + GPU)

Transcreve reuniões, aulas e entrevistas em português usando o modelo **Whisper large-v3** na GPU gratuita do Google Colab.

**Como usar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
2. Coloque o áudio ou vídeo numa pasta do seu Google Drive (padrão: `MyDrive/TRANSCRITOR`).
3. Ajuste o nome do arquivo na célula **2** e rode as células em ordem.

Saídas (salvas na mesma pasta do Drive): `<nome>_transcricao.txt` com marcação de tempo e `<nome>.srt` (legenda).

In [ ]:
# 1. INSTALAÇÃO
!apt-get install -y ffmpeg -q
!pip install faster-whisper -q

In [ ]:
# 2. CONFIGURAÇÃO: conecta o Drive e define o arquivo
from google.colab import drive
drive.mount('/content/drive')

PASTA = "/content/drive/MyDrive/TRANSCRITOR"   # pasta do Drive com o áudio
ARQUIVO = "reuniao.mp4"                         # nome do áudio/vídeo dentro da pasta
MODELO = "large-v3"                             # tiny, base, small, medium ou large-v3
INITIAL_PROMPT = "Transcrição em português do Brasil."   # dica: inclua nomes próprios e siglas do assunto

import os
ENTRADA = os.path.join(PASTA, ARQUIVO)
NOME = os.path.splitext(ARQUIVO)[0]
assert os.path.exists(ENTRADA), f"Arquivo não encontrado: {ENTRADA}"
print("OK:", ENTRADA)

In [ ]:
# 3. LIMPEZA DO ÁUDIO (ffmpeg)
# mono, 16 kHz (o formato do Whisper), corta frequências fora da voz,
# reduz ruído de fundo e normaliza o volume. O arquivo original não é alterado.
AUDIO_LIMPO = os.path.join(PASTA, f"{NOME}_limpo.wav")
!ffmpeg -y -loglevel error -i "{ENTRADA}" -vn -ac 1 -ar 16000 -af "highpass=f=100,lowpass=f=8000,afftdn=nf=-25,loudnorm" "{AUDIO_LIMPO}"
print("Áudio limpo:", AUDIO_LIMPO)

In [ ]:
# 4. TRANSCRIÇÃO NA GPU
import time
from faster_whisper import WhisperModel

def tempo_srt(seg):
    ms = int(round(seg * 1000)); h, ms = divmod(ms, 3_600_000); m, ms = divmod(ms, 60_000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

print(f"Carregando o modelo '{MODELO}' na GPU...")
model = WhisperModel(MODELO, device="cuda", compute_type="float16")

inicio = time.time()
segmentos, info = model.transcribe(
    AUDIO_LIMPO,
    language="pt",
    beam_size=10,
    best_of=5,
    temperature=0.0,
    condition_on_previous_text=False,   # um erro não "contamina" os trechos seguintes
    vad_filter=True,                    # ignora silêncio e ruído, reduz alucinações
    vad_parameters=dict(min_silence_duration_ms=500),
    compression_ratio_threshold=2.4,    # descarta trechos repetitivos suspeitos
    log_prob_threshold=-1.0,            # descarta trechos de baixa confiança
    no_speech_threshold=0.6,            # não "inventa" fala sobre ruído
    initial_prompt=INITIAL_PROMPT,
)
print(f"Duração: {info.duration/60:.1f} min | idioma: {info.language} ({info.language_probability:.2f})\n")

TXT = os.path.join(PASTA, f"{NOME}_transcricao.txt")
SRT = os.path.join(PASTA, f"{NOME}.srt")
with open(TXT, "w", encoding="utf-8") as f_txt, open(SRT, "w", encoding="utf-8") as f_srt:
    for n, seg in enumerate(segmentos, start=1):
        texto = seg.text.strip()
        linha = f"[{seg.start:.1f}s -> {seg.end:.1f}s] {texto}"
        print(linha)
        f_txt.write(linha + "\n")
        f_srt.write(f"{n}\n{tempo_srt(seg.start)} --> {tempo_srt(seg.end)}\n{texto}\n\n")

gasto = time.time() - inicio
print(f"\n✅ Pronto em {gasto/60:.1f} min ({info.duration/gasto:.1f}x a velocidade do áudio)")
print("Arquivos salvos no Drive:", TXT, "|", SRT)

## Dicas
- **Nomes próprios e siglas** saem melhores se você colocá-los no `INITIAL_PROMPT` (ex.: `"Reunião do escritório. Termos: balancete, SPED, ECD, CRC."`).
- **Áudio longo** (mais de 1 hora) funciona normalmente; a sessão gratuita do Colab tem limite de tempo, então mantenha a aba aberta.
- **Privacidade:** o áudio fica no seu Drive e é processado na sua sessão do Colab; nada é enviado para serviços de transcrição de terceiros.